# TopicGPT: Topic Continuity Rate

**Scenario 3b** — Best-match assignment with mismatch detection.

| Priority | Category | Condition |
|----------|----------|-----------|
| 1 | **Disappear** | sim ≤ 0.2 |
| 2 | **Merge** | sim > 0.2, and the t+1-topic has >1 source |
| 3 | **Mismatch** | sim > 0.2, single source, best-match ID ≠ own topic ID |
| 4 | **Stable** | sim > 0.6, best-match ID = own topic ID |
| 5 | **Evolve** | 0.2 < sim ≤ 0.6, best-match ID = own topic ID |
| — | **New** | t+1-topic has no incoming source with sim > 0.2 |


From t-side: Disappear + Merge + Mismatch + Stable + Evolve = 100%


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
TEMPORAL_DIR = Path("../../../../results/topicGpt/temporal")
RESULT_DIR = Path("../../../../results/topicGpt/consistency")
THRESHOLD_DISAPPEAR = 0.2  # sim <= 0.2 → disappear
THRESHOLD_STABLE    = 0.6  # sim > 0.5  → stable (if same ID)

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Threshold Disappear : sim <= {THRESHOLD_DISAPPEAR}")
print(f"Threshold Stable    : sim > {THRESHOLD_STABLE}")
print(f"Reading from: {TEMPORAL_DIR}")
print(f"Saving to: {RESULT_DIR}")


Threshold Disappear : sim <= 0.2
Threshold Stable    : sim > 0.6
Reading from: ../../../../results/topicGpt/temporal
Saving to: ../../../../results/topicGpt/consistency


In [3]:
def parse_words(words_str):
    return [w.strip() for w in str(words_str).split(",")]



def rbo(list1, list2, p=0.9):
    if not list1 and not list2:
        return 1.0
    if not list1 or not list2:
        return 0.0

    # assign short (S) and long (L)
    if len(list1) <= len(list2):
        S, L = list1, list2
    else:
        S, L = list2, list1

    s, l = len(S), len(L)

    S_seen = set()
    L_seen = set()

    X = 0  # overlap
    rbo = 0.0
    disjoint = 0.0
    ext_term = 0.0

    for d in range(l):
        if d < s:
            s_item = S[d]
            S_seen.add(s_item)
        else:
            s_item = None

        l_item = L[d]
        L_seen.add(l_item)

        overlap_incr = 0

        if d < s:
            if s_item == l_item:
                overlap_incr = 1
            else:
                if s_item in L_seen:
                    overlap_incr += 1
                if l_item in S_seen:
                    overlap_incr += 1
        else:
            if l_item in S_seen:
                overlap_incr = 1

        X += overlap_incr

        if d < s:
            A_d = 2.0 * X / (len(S_seen) + len(L_seen))
        else:
            A_d = X / (d + 1)

        rbo += (1 - p) * (p ** d) * A_d

        if d < s:
            ext_term = A_d * (p ** (d + 1))
        else:
            X_s = X - overlap_incr if d == s else X_s
            disjoint += (1 - p) * (p ** d) * (
                X_s * (d + 1 - s) / ((d + 1) * s)
            )
            ext_term = (
                ((X - X_s) / (d + 1) + X_s / s)
                * (p ** (d + 1))
            )

        # optional optimization (safe)
        if p ** d < 1e-12:
            break

    return min(max(rbo + disjoint + ext_term, 0.0), 1.0)

## Compute Continuity Rate (Best-Match)

In [4]:
for subject in LIST_SUBJECT:
    print(f"\n'======================================================================'")
    print(f"Continuity Rate: {subject.upper()} (TopicGPT)")
    print(f"'======================================================================'")

    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")

    topic_words = {}
    for _, row in evo_df.iterrows():
        key = (int(row["year"]), int(row["topic_id"]))
        topic_words[key] = parse_words(row["top_words"])

    years = sorted(evo_df["year"].unique())

    all_transition_rows = []
    all_merge_rows = []
    all_new_rows = []
    summary_rows = []

    for i in range(len(years) - 1):
        t, t_next = int(years[i]), int(years[i + 1])

        topics_t  = sorted([tid for (y, tid) in topic_words if y == t])
        topics_t1 = sorted([tid for (y, tid) in topic_words if y == t_next])

        if not topics_t or not topics_t1:
            continue

        # Build full RBO similarity matrix (t → t+1)
        sim_matrix = np.zeros((len(topics_t), len(topics_t1)))
        for ii, tid_t in enumerate(topics_t):
            words_t = topic_words.get((t, tid_t), [])
            for jj, tid_t1 in enumerate(topics_t1):
                words_t1_tmp = topic_words.get((t_next, tid_t1), [])
                sim_matrix[ii, jj] = rbo(words_t, words_t1_tmp, p=0.9)

        # ── Step 1: Each t-topic → its best t+1-topic ────────────────────────
        best_match_idx = np.argmax(sim_matrix, axis=1)
        best_match_sim = np.max(sim_matrix, axis=1)

        # ── Step 2: Count sources (sim > 0.2) per t+1-topic ──────────────────
        target_counts = Counter()
        for idx in range(len(topics_t)):
            if float(best_match_sim[idx]) > THRESHOLD_DISAPPEAR:
                target_counts[topics_t1[best_match_idx[idx]]] += 1

        # merge_targets: t+1-topics claimed by >1 source with sim > 0.2
        merge_targets = {tgt for tgt, cnt in target_counts.items() if cnt > 1}

        # ── Step 3: Classify each t-topic ─────────────────────────────────────
        n_stable = n_evolve = n_merge = n_disappear = n_mismatch = 0

        for idx, tid in enumerate(topics_t):
            sim_val    = float(best_match_sim[idx])
            target_tid = topics_t1[best_match_idx[idx]]
            words_t    = ", ".join(topic_words.get((t, tid), []))

            # ── 1. DISAPPEAR: sim <= 0.2 ──────────────────────────────────────
            if sim_val <= THRESHOLD_DISAPPEAR:
                category = "disappear"
                n_disappear += 1

            # ── 2. MERGE: sim > 0.2 AND target has >1 source ─────────────────
            elif tid in merge_targets:
                tar_id = tid
                sources = [tid for idx, tid in enumerate(topics_t)
                       if topics_t1[best_match_idx[idx]] == tar_id
                       and float(best_match_sim[idx]) > THRESHOLD_DISAPPEAR]
                if tid in sources:
                    category = "merge"
                    n_merge += 1
                else:
                    category = "mismatch"
                    n_merge += 1

            # ── 3. MISMATCH: sim > 0.2, single source, target ID ≠ own ID ────
            elif target_tid != tid:
                category   = "mismatch"
                n_mismatch += 1

            # ── 4. STABLE: sim > 0.5, target ID = own ID ─────────────────────
            elif sim_val > THRESHOLD_STABLE:
                category = "stable"
                n_stable += 1

            # ── 5. EVOLVE: 0.2 < sim <= 0.5, target ID = own ID ─────────────
            else:
                category = "evolve"
                n_evolve += 1

            all_transition_rows.append({
                "subject": subject, "year_from": t, "year_to": t_next,
                "topic_id": tid, "category": category,
                "best_match_topic": target_tid,
                "best_match_sim": round(sim_val, 6),
                "words": words_t,
            })

        # ── Step 4: Record merge groups ────────────────────────────────────────
        for target_tid in merge_targets:
            sources = [tid for idx, tid in enumerate(topics_t)
                       if topics_t1[best_match_idx[idx]] == target_tid
                       and float(best_match_sim[idx]) > THRESHOLD_DISAPPEAR]
            source_sims = [round(float(sim_matrix[topics_t.index(s),
                                                   topics_t1.index(target_tid)]), 4)
                           for s in sources]
            words_t1 = ", ".join(topic_words.get((t_next, target_tid), []))
            all_merge_rows.append({
                "subject": subject, "year_from": t, "year_to": t_next,
                "target_topic": target_tid,
                "n_sources": len(sources),
                "source_topics": str(sources),
                "source_sims": str(source_sims),
                "target_words": words_t1,
            })

        # ── Step 5: New topics — no incoming source with sim > 0.2 ────────────
        matched_t1 = set(topics_t[idx]
                         for idx in range(len(topics_t))
                         if float(best_match_sim[idx]) > THRESHOLD_DISAPPEAR)
        new_topics = [tid for tid in topics_t1 if tid not in matched_t1]
        for tid in new_topics:
            words_t1 = ", ".join(topic_words.get((t_next, tid), []))
            all_new_rows.append({
                "subject": subject, "year_from": t, "year_to": t_next,
                "topic_id": tid, "words": words_t1,
            })

        total_t        = len(topics_t)
        n_new          = len(new_topics)
        n_merge_groups = len(merge_targets)

        summary_rows.append({
            "subject": subject, "year_from": t, "year_to": t_next,
            "n_topics_t": total_t, "n_topics_t1": len(topics_t1),
            "n_stable": n_stable, "n_evolve": n_evolve, "n_merge": n_merge,
            "n_mismatch": n_mismatch, "n_disappear": n_disappear,
            "n_merge_groups": n_merge_groups, "n_new": n_new,
            "pct_stable":    round(n_stable    / total_t * 100, 2),
            "pct_evolve":    round(n_evolve    / total_t * 100, 2),
            "pct_merge":     round(n_merge     / total_t * 100, 2),
            "pct_mismatch":  round(n_mismatch  / total_t * 100, 2),
            "pct_disappear": round(n_disappear / total_t * 100, 2),
        })

        print(f"  {t}→{t_next}: Stable={n_stable} ({n_stable/total_t:.0%})  "
              f"Evolve={n_evolve} ({n_evolve/total_t:.0%})  "
              f"Merge={n_merge} ({n_merge/total_t:.0%})  "
              f"Mismatch={n_mismatch} ({n_mismatch/total_t:.0%})  "
              f"Disappear={n_disappear} ({n_disappear/total_t:.0%})  "
              f"New={n_new}")

    # Save CSVs
    pd.DataFrame(all_transition_rows).to_csv(
        RESULT_DIR / subject / "continuity_transitions.csv", index=False)
    pd.DataFrame(all_merge_rows).to_csv(
        RESULT_DIR / subject / "continuity_merges.csv", index=False)
    pd.DataFrame(all_new_rows).to_csv(
        RESULT_DIR / subject / "continuity_new_topics.csv", index=False)

    sum_df = pd.DataFrame(summary_rows)
    sum_df.to_csv(RESULT_DIR / subject / "continuity_summary.csv", index=False)

    subj_sum = sum_df[sum_df["subject"] == subject]
    overall = {
        "subject": subject,
        "threshold_disappear": THRESHOLD_DISAPPEAR,
        "threshold_stable":    THRESHOLD_STABLE,
        "avg_pct_stable":    round(subj_sum["pct_stable"].mean(),    2),
        "avg_pct_evolve":    round(subj_sum["pct_evolve"].mean(),    2),
        "avg_pct_merge":     round(subj_sum["pct_merge"].mean(),     2),
        "avg_pct_mismatch":  round(subj_sum["pct_mismatch"].mean(),  2),
        "avg_pct_disappear": round(subj_sum["pct_disappear"].mean(), 2),
        "total_merge_groups": int(subj_sum["n_merge_groups"].sum()),
        "total_new":          int(subj_sum["n_new"].sum()),
    }
    pd.DataFrame([overall]).to_csv(
        RESULT_DIR / subject / "continuity_overall.csv", index=False)

    print(f"\n  Overall: Stable={overall['avg_pct_stable']:.1f}%  "
          f"Evolve={overall['avg_pct_evolve']:.1f}%  "
          f"Merge={overall['avg_pct_merge']:.1f}%  "
          f"Mismatch={overall['avg_pct_mismatch']:.1f}%  "
          f"Disappear={overall['avg_pct_disappear']:.1f}%  "
          f"New={overall['total_new']}")
    print(f"  Saved: {RESULT_DIR / subject}")



'======================================================================'
Continuity Rate: CS (TopicGPT)
'======================================================================'
  2000→2001: Stable=6 (24%)  Evolve=10 (40%)  Merge=1 (4%)  Mismatch=1 (4%)  Disappear=7 (28%)  New=18
  2001→2002: Stable=9 (26%)  Evolve=13 (37%)  Merge=0 (0%)  Mismatch=0 (0%)  Disappear=13 (37%)  New=13
  2002→2003: Stable=7 (20%)  Evolve=13 (37%)  Merge=0 (0%)  Mismatch=0 (0%)  Disappear=15 (43%)  New=11
  2003→2004: Stable=4 (13%)  Evolve=10 (32%)  Merge=4 (13%)  Mismatch=4 (13%)  Disappear=9 (29%)  New=14
  2004→2005: Stable=9 (26%)  Evolve=16 (46%)  Merge=1 (3%)  Mismatch=1 (3%)  Disappear=8 (23%)  New=12
  2005→2006: Stable=7 (18%)  Evolve=17 (44%)  Merge=2 (5%)  Mismatch=3 (8%)  Disappear=10 (26%)  New=11
  2006→2007: Stable=7 (18%)  Evolve=12 (32%)  Merge=3 (8%)  Mismatch=3 (8%)  Disappear=13 (34%)  New=16
  2007→2008: Stable=10 (24%)  Evolve=18 (44%)  Merge=1 (2%)  Mismatch=1 (2%)  Disappear=11 (27%

## Analysis Review

In [5]:
for subject in LIST_SUBJECT:
    print(f"\n'======================================================================'")
    print(f"Analysis Review: {subject.upper()} (TopicGPT)")
    print(f"'======================================================================'")

    trans_df   = pd.read_csv(RESULT_DIR / subject / "continuity_transitions.csv")
    merge_path = RESULT_DIR / subject / "continuity_merges.csv"
    merge_df   = pd.read_csv(merge_path) if merge_path.stat().st_size > 10 else pd.DataFrame()
    new_path   = RESULT_DIR / subject / "continuity_new_topics.csv"
    new_df     = pd.read_csv(new_path)   if new_path.stat().st_size   > 10 else pd.DataFrame()
    evo_df     = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")

    # --- MERGE ---
    merged = trans_df[trans_df["category"] == "merge"]
    print(f"\n  🔗 MERGE (sim > {THRESHOLD_DISAPPEAR}, >1 source): {len(merged)} events")
    if len(merge_df) > 0:
        print("  Top 10 merges (most sources):")
        for _, r in merge_df.nlargest(min(10, len(merge_df)), "n_sources").iterrows():
            w = str(r["target_words"])[:40]
            print(f"    T{int(r['target_topic']):>3} @ {int(r['year_to'])} "
                  f"← {r['source_topics']} (sims={r['source_sims']}) | {w}")

    # --- MISMATCH ---
    mismatch = trans_df[trans_df["category"] == "mismatch"]
    print(f"\n  ⚠️  MISMATCH (sim > {THRESHOLD_DISAPPEAR}, target ID ≠ own ID): {len(mismatch)} events")
    if len(mismatch) > 0:
        per_year = mismatch.groupby("year_from").size()
        print("  Per year:")
        for yr, cnt in per_year.items():
            print(f"    {int(yr)}: {cnt} topics")
        print("  Sample (up to 5):")
        for _, r in mismatch.head(5).iterrows():
            print(f"    T{int(r['topic_id']):>3} @ {int(r['year_from'])}→{int(r['year_to'])} "
                  f"best→T{int(r['best_match_topic'])} (sim={r['best_match_sim']:.3f}) | {str(r['words'])[:50]}")

    # --- STABLE ---
    stable = trans_df[trans_df["category"] == "stable"]
    print(f"\n  🟢 STABLE (sim > {THRESHOLD_STABLE}, same ID): {len(stable)} events")
    if len(stable) > 0:
        per_year = stable.groupby("year_from").size()
        print("  Per year:")
        for yr, cnt in per_year.items():
            print(f"    {int(yr)}: {cnt} topics")

    # --- EVOLVE ---
    evolved = trans_df[trans_df["category"] == "evolve"]
    print(f"\n  🔄 EVOLVE ({THRESHOLD_DISAPPEAR} < sim <= {THRESHOLD_STABLE}, same ID): {len(evolved)} events")
    if len(evolved) > 0:
        per_year = evolved.groupby("year_from").size()
        print("  Per year:")
        for yr, cnt in per_year.items():
            print(f"    {int(yr)}: {cnt} topics")

    # --- DISAPPEAR ---
    disappeared = trans_df[trans_df["category"] == "disappear"]
    print(f"\n  📉 DISAPPEARED (sim <= {THRESHOLD_DISAPPEAR}): {len(disappeared)} events")
    if len(disappeared) > 0:
        per_year = disappeared.groupby("year_from").size()
        print("  Per year:")
        for yr, cnt in per_year.items():
            print(f"    {int(yr)}: {cnt} topics")
        perm = []
        for _, r in disappeared.iterrows():
            tid   = int(r["topic_id"])
            later = evo_df[(evo_df["topic_id"] == tid) & (evo_df["year"] > int(r["year_to"]))]
            if len(later) == 0:
                perm.append(f"    T{tid:>3} last @ {int(r['year_from'])} (sim={r['best_match_sim']:.3f}) | {str(r['words'])[:50]}")
        print(f"\n  Permanently gone: {len(perm)}")
        for line in perm[:10]:
            print(line)
        if len(perm) > 10:
            print(f"    ... +{len(perm)-10} more")

    # --- NEW TOPICS ---
    print(f"\n  🆕 NEW TOPICS (no incoming sim > {THRESHOLD_DISAPPEAR}): {len(new_df)} events")
    if len(new_df) > 0:
        per_year = new_df.groupby("year_to").size()
        print("  Per year:")
        for yr, cnt in per_year.items():
            print(f"    {int(yr)}: {cnt} new topics")

    # --- STABILITY RANKING ---
    print("\n  🏆 STABILITY RANKING:")
    ts = trans_df.groupby("topic_id")["category"].value_counts().unstack(fill_value=0)
    for cat in ["stable", "evolve", "merge", "mismatch", "disappear"]:
        if cat not in ts.columns:
            ts[cat] = 0
    ts["total"] = ts.sum(axis=1)
    ts["stability_pct"] = (ts["stable"] / ts["total"] * 100).round(1)
    ts = ts.sort_values("stability_pct", ascending=False)
    print("  Top 5 most stable:")
    for tid, r in ts.head(5).iterrows():
        print(f"    T{int(tid):>3} | {r['stability_pct']:.0f}% stable "
              f"({int(r['stable'])}S {int(r.get('evolve',0))}Ev "
              f"{int(r.get('merge',0))}M {int(r.get('mismatch',0))}Mm {int(r.get('disappear',0))}D)")
    print("  Top 5 most unstable:")
    for tid, r in ts.tail(5).iterrows():
        print(f"    T{int(tid):>3} | {r['stability_pct']:.0f}% stable "
              f"({int(r['stable'])}S {int(r.get('evolve',0))}Ev "
              f"{int(r.get('merge',0))}M {int(r.get('mismatch',0))}Mm {int(r.get('disappear',0))}D)")
    print()



'======================================================================'
Analysis Review: CS (TopicGPT)
'======================================================================'

  🔗 MERGE (sim > 0.2, >1 source): 49 events
  Top 10 merges (most sources):
    T 62 @ 2001 ← [2, 62] (sims=[0.2703, 0.6138]) | document, indicative, summary, summariza
    T 10 @ 2004 ← [10, 13] (sims=[0.6915, 0.2703]) | language, logic, programming, chr, progr
    T 27 @ 2004 ← [27, 48] (sims=[0.6671, 0.2703]) | protocol, security, soap, principal, aut
    T 12 @ 2004 ← [12, 28] (sims=[0.3486, 0.2274]) | coroutine, graph, expander, bit, obdd, v
    T 44 @ 2004 ← [44, 61] (sims=[0.5946, 0.4121]) | game, player, barney, wilma, digraph, st
    T 31 @ 2005 ← [16, 31] (sims=[0.3073, 0.5033]) | graph, dominating, domatic, algorithm, e
    T 33 @ 2006 ← [33, 45] (sims=[0.2709, 0.3274]) | network, community, agent, structure, za
    T 41 @ 2006 ← [41, 101] (sims=[0.3943, 0.2703]) | code, mrd, bch, pseudocodeword, he